In [ ]:
!pip install hmmlearn
import numpy as np
from hmmlearn import hmm

# 1. Define Chain 1: Voice Activity (Hidden States: Silent=0, Speaking=1)
model_voice = hmm.GaussianHMM(n_components=2, covariance_type="full")
model_voice.startprob_ = np.array([0.5, 0.5])
model_voice.transmat_ = np.array([[0.9, 0.1],
                                  [0.2, 0.8]])
model_voice.means_ = np.array([[0.0], [10.0]]) # Silent is 0dB, Speaking is 10dB
model_voice.covars_ = np.array([[[1.0]], [[1.0]]]) # Initialize covariance for each state

# 2. Define Chain 2: Physical Motion (Hidden States: Still=0, Moving=1)
model_motion = hmm.GaussianHMM(n_components=2, covariance_type="full")
model_motion.startprob_ = np.array([0.5, 0.5])
model_motion.transmat_ = np.array([[0.7, 0.3],
                                   [0.4, 0.6]])
model_motion.means_ = np.array([[0.0], [5.0]]) # Still is 0 units, Moving is 5 units
model_motion.covars_ = np.array([[[1.0]], [[1.0]]]) # Initialize covariance for each state

# 3. The "Factorial" Observation (The Fusion)
# In FHMM, the observation is the SUM (or combination) of the hidden states
def generate_fhmm_data(n_steps=20):
    # Sample from both chains independently
    v_states, v_obs = model_voice.sample(n_steps)
    m_states, m_obs = model_motion.sample(n_steps)

    # Combined Observation (This is what the sensor "sees")
    total_observation = v_obs + m_obs

    return total_observation, v_states, m_states

# 4. Running the simulation
obs, v_true, m_true = generate_fhmm_data(10)

print("Time Step | Voice State | Motion State | Combined Observation")
print("-" * 60)
for i in range(10):
    print(f"   {i}      |      {v_true[i][0]:.0f}      |      {m_true[i][0]:.0f}       |      {obs[i]:.2f}")